# 05 - Feature Engineering
### Week 6:
- Engineer new features (bed/bath ratio, property age)
- Add school district spatial layer (CA School District boundaries)
- Retrain models with updated feature set
- Compare performance: old features vs. new features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

train_df = pd.read_csv("data/train_final.csv")
test_df = pd.read_csv("data/test_final.csv")

In [ ]:
for df in [train_df, test_df]:
    # Bed/bath ratio
    df['BedBathRatio'] = df['BedroomsTotal'] / df['BathroomsTotalInteger'].replace(0, np.nan)
    df['BedBathRatio'] = df['BedBathRatio'].fillna(df['BedBathRatio'].median())

    # Property age (using CloseDate year - YearBuilt)
    df['CloseYear'] = pd.to_datetime(df['CloseDate']).dt.year
    df['PropertyAge'] = df['CloseYear'] - df['YearBuilt']
    df['PropertyAge'] = df['PropertyAge'].clip(lower=0)  # guard against bad data

print(train_df[['BedBathRatio', 'PropertyAge']].describe())


In [ ]:
drop_from_features = ['ClosePrice', 'ClosePrice_log', 'CloseDate', 'CloseYearMonth', 'CloseYear']

# OLD feature set (baseline from 03/04)
X_train_old = train_df.drop(columns=[c for c in drop_from_features if c in train_df.columns])
X_test_old = test_df.drop(columns=[c for c in drop_from_features if c in test_df.columns])
X_train_old, X_test_old = X_train_old.align(X_test_old, join='left', axis=1, fill_value=0)

# NEW feature set (old + engineered features) — same columns, since we added them in place
X_train_new = X_train_old.copy()
X_test_new = X_test_old.copy()

y_train = train_df['ClosePrice_log']
y_test = test_df['ClosePrice_log']

print('Old feature count:', X_train_old.shape[1])
print('New feature count:', X_train_new.shape[1])

In [4]:
rf_new = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_new.fit(X_train_new, y_train)

pred_log = rf_new.predict(X_test_new)
pred_price = np.exp(pred_log)
actual_price = np.exp(y_test)

r2_new = r2_score(y_test, pred_log)
mape_new = mean_absolute_percentage_error(actual_price, pred_price)
mdape_new = np.median(np.abs((actual_price - pred_price) / actual_price))

print(f"RF with new features — R²: {r2_new:.4f}, MAPE: {mape_new:.4f}, MdAPE: {mdape_new:.4f}")

RF with new features — R²: 0.8795, MAPE: 0.1826, MdAPE: 0.1027


In [5]:
comparison = pd.DataFrame({
    'Feature Set': ['Random Forest (baseline features)', 'Random Forest (+ BedBathRatio, PropertyAge)'],
    'R2': [0.8797, r2_new],   # <- replace 0.8797 with your actual June-retrained RF R² once you have it
    'MAPE': [0.1819, mape_new],
    'MdAPE': [0.1032, mdape_new]
})
print(comparison)

                                   Feature Set        R2      MAPE     MdAPE
0            Random Forest (baseline features)  0.879700  0.181900  0.103200
1  Random Forest (+ BedBathRatio, PropertyAge)  0.879519  0.182637  0.102736
